# 13 - Python Interview Gotchas: Rapid-Fire FAQ (Capstone)

Concise, interview/revision focused. Closes out the Python subchapter. Format: predict the output, then run the cell to check yourself.

## 1. The mutable default argument trap

The classic. A default argument is evaluated ONCE, at function definition time, not on every call -- a mutable default is shared and accumulates across calls.

In [1]:
def append_to(item, bucket=[]):   # bucket is created ONCE, when the function is defined
    bucket.append(item)
    return bucket

print(append_to(1))
print(append_to(2))
print(append_to(3))   # NOT [3] -- keeps growing across unrelated calls

# fix: use None as the default, create fresh inside
def append_to_fixed(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

print(append_to_fixed(1))
print(append_to_fixed(2))

[1]
[1, 2]
[1, 2, 3]
[1]
[2]


## 2. `is` vs `==`

`==` compares VALUE (calls `__eq__`); `is` compares IDENTITY (same object in memory). CPython caches small integers (-5 to 256) and some strings, which makes `is` "accidentally" work on them -- relying on that is a bug waiting to happen outside that range.

In [2]:
a = 256
b = 256
print("256 is 256:", a is b)      # True -- small int cache

a = 257
b = 257
print("257 is 257:", a is b)      # often False -- outside the cached range (implementation detail!)

x = [1, 2]
y = [1, 2]
print("equal value:", x == y)      # True
print("same object:", x is y)      # False -- two distinct list objects
print("same object:", x is x)      # True

print("None comparisons always use is:", x is None)   # `is None`, never `== None`, is the convention

256 is 256: True
257 is 257: False
equal value: True
same object: False
same object: True
None comparisons always use is: False


## 3. Shallow copy vs deep copy

`copy.copy()` / `list(x)` / `x[:]` copy the outer container only -- nested mutable objects are still SHARED. `copy.deepcopy()` recursively copies everything.

In [3]:
import copy

original = [[1, 2], [3, 4]]
shallow = copy.copy(original)
deep = copy.deepcopy(original)

original[0].append(99)   # mutate a NESTED list

print("shallow sees the change (shares inner lists):", shallow)
print("deep does NOT (fully independent):           ", deep)

shallow sees the change (shares inner lists): [[1, 2, 99], [3, 4]]
deep does NOT (fully independent):            [[1, 2], [3, 4]]


## 4. Late binding closures in a loop

Closures capture the VARIABLE, not its value at the time the lambda was created -- by the time any of them run, the loop has finished and the variable holds its final value.

In [4]:
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])              # [2, 2, 2], not [0, 1, 2]

funcs_fixed = [lambda i=i: i for i in range(3)]   # default arg forces evaluation NOW
print([f() for f in funcs_fixed])                  # [0, 1, 2]

[2, 2, 2]
[0, 1, 2]


## 5. Exception variables vanish after the except block

Python 3 deletes the `as e` name when the except block ends (to break a reference cycle with the traceback) -- using it afterward raises `NameError`, even though it clearly "existed" a line earlier.

In [5]:
try:
    1 / 0
except ZeroDivisionError as e:
    print("inside the block:", e)

try:
    print("outside the block:", e)
except NameError as err:
    print("NameError outside the block:", err)

inside the block: division by zero
NameError outside the block: name 'e' is not defined


## 6. Floating point is not exact

Binary floating point cannot represent 0.1 or 0.2 exactly -- comparing floats with `==` is a common, quiet bug. Compare with a tolerance instead.

In [6]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)          # False

import math
print(math.isclose(0.1 + 0.2, 0.3))   # True -- the correct way to compare floats

0.30000000000000004
False
True


## 7. Chained comparisons are not what they look like in most languages

`a < b < c` means `(a < b) and (b < c)` -- both share the middle term -- NOT `(a < b) < c`, which is what naive translation from other languages would suggest.

In [7]:
print(1 < 2 < 3)          # True: 1<2 AND 2<3
print(1 < 3 < 2)          # False: 1<3 True, but 3<2 False

# the "translated from another language" trap:
result = (1 < 2) < 3      # True < 3 -> 1 < 3 -> True (True behaves as 1) -- coincidentally same answer here
print(result, "-- coincidentally matches, but the MEANING is different; do not rely on this")

True
False
True -- coincidentally matches, but the MEANING is different; do not rely on this


## 8. Mutating a list inside a "immutable" tuple

A tuple is immutable in the sense that it cannot be made to point to DIFFERENT objects -- but if one of those objects is itself mutable, mutating it in place is still allowed.

In [8]:
t = (1, 2, [3, 4])

try:
    t[2] = [99]           # rebinding the slot itself -- blocked
except TypeError as e:
    print("blocked:", e)

t[2].append(5)             # mutating the list THAT SLOT POINTS TO -- allowed
print("mutated in place anyway:", t)

blocked: 'tuple' object does not support item assignment
mutated in place anyway: (1, 2, [3, 4, 5])


## 9. Class variable vs instance variable shadowing

Reading `self.x` finds the class variable if no instance variable exists yet; ASSIGNING `self.x = ...` always creates a new INSTANCE variable, which then shadows the class one for that object only, leaving it unchanged for all other instances.

In [9]:
class Counter:
    count = 0                # class variable, shared

    def increment(self):
        self.count += 1       # reads class var (0), then creates an INSTANCE var count=1

a, b = Counter(), Counter()
a.increment()
a.increment()
print("a.count:", a.count, "| b.count:", b.count, "| Counter.count:", Counter.count)
# a has its OWN count now (2); b still falls through to the unchanged class variable (0)

a.count: 2 | b.count: 0 | Counter.count: 0


## 10. `global` vs `nonlocal`

`global` reaches all the way to module scope. `nonlocal` reaches the nearest ENCLOSING function scope (for closures) -- neither is needed just to READ an outer variable, only to REASSIGN it.

In [10]:
x = "module level"

def outer():
    x = "enclosing level"
    def inner_read():
        print("read without any keyword:", x)   # reading works fine, no keyword needed
    def inner_write():
        nonlocal x
        x = "changed by inner_write"
    inner_read()
    inner_write()
    print("after nonlocal write:", x)

outer()
print("module-level x unaffected:", x)

read without any keyword: enclosing level
after nonlocal write: changed by inner_write
module-level x unaffected: module level


## Rapid recap table

| Gotcha | One-line fix |
|---|---|
| Mutable default arg | default `None`, create the mutable inside the function |
| `is` vs `==` | `==` for value, `is` only for identity / `None` checks |
| Shallow vs deep copy | `copy.deepcopy()` when nested mutables must not be shared |
| Late-binding closures | `lambda x=x: ...` to capture the value now |
| Exception var scope | assign `e` to another name inside the block if needed later |
| Float equality | `math.isclose()`, never bare `==` |
| Chained comparisons | fine to use, just know both sides share the middle term |
| Tuple of mutables | tuple blocks reassignment, not in-place mutation of its contents |
| Class vs instance attrs | assignment always creates/shadows on the INSTANCE |
| global/nonlocal | only required to REASSIGN an outer-scope name, not to read it |

## Practice

Predict the output of each before running, then check:

```python
# 1
def f(a, items=[]):
    items.append(a)
    return items
print(f(1), f(2))

# 2
class A:
    shared = []
    def __init__(self, x):
        self.shared.append(x)   # note: .append, not =
a, b = A(1), A(2)
print(a.shared, b.shared)

# 3
print([1, 2, 3] is [1, 2, 3])
print((1, 2) is (1, 2))
```

In [11]:
# Check your predictions here
def f(a, items=[]):
    items.append(a)
    return items
print(f(1), f(2))

class A:
    shared = []
    def __init__(self, x):
        self.shared.append(x)
a, b = A(1), A(2)
print(a.shared, b.shared)

print([1, 2, 3] is [1, 2, 3])
print((1, 2) is (1, 2))

[1, 2] [1, 2]
[1, 2] [1, 2]
False
True


<>:15: SyntaxWarning: "is" with 'tuple' literal. Did you mean "=="?
<>:15: SyntaxWarning: "is" with 'tuple' literal. Did you mean "=="?
/tmp/ipykernel_1125/3265252955.py:15: SyntaxWarning: "is" with 'tuple' literal. Did you mean "=="?
  print((1, 2) is (1, 2))
